# Solving Nurse Rostering Problem with SAT by Sequential Encoding for At Least K Constraints

Giới thiệu một chút là sequence constraints (ràng buộc chuỗi) có thể được hiểu là trong một đoạn liên tiếp với độ dài $w$ thì số lần xuất hiện của một điều kiện phải nằm trong một giới hạn nào đó. Ví dụ, trong một chuỗi liên tiếp của các ca làm việc, có thể yêu cầu rằng một nhân viên phải làm việc ít nhất $k$ lần trong mỗi chuỗi liên tiếp có độ dài $w$. Đây là một loại ràng buộc phổ biến trong các bài toán lập lịch, đặc biệt là trong bài toán lập lịch nhân viên (Nurse Rostering Problem - NRP).

## At Least Sequence Constraints
### Định nghĩa
Cho một dãy biến Boolean $\Omega = (x_1, x_2, \ldots, x_n)$ và một số nguyên dương $w$ (độ dài chuỗi) và $k$ (số lần xuất hiện tối thiểu), ràng buộc "at least $k$ in every sequence of length $w$" yêu cầu rằng trong mỗi chuỗi con liên tiếp của $\Omega$ có độ dài $w$, phải có ít nhất $k$ biến được gán giá trị True.
$$
\textbf{ALSC}(\Omega,w) = \bigwedge_{i=0}^{n-w} \left( \sum_{j=i+1}^{i+w} x_j \geq k \right), \quad 1 \leq k < w
$$

<center>
    <img src="./images/ALSC/fig1.png" alt="ALSC Example" width="600">
    <p>Hình minh họa ràng buộc ALSC với w=4.</p>
</center>   

## Phân rã ALSC và mã hoá ALSC bằng SCL

**Tính chất 1**: Ràng buộc $x_1+x_2+\ldots+x_w \geq k$ có thể được phân rã thành một tập hợp các ràng buộc con như sau:
$$
\forall m \in \{1,2,\ldots,w-1\}: \quad
\bigwedge_{i=1}^{k} \left(\left(\sum_{j=1}^{m} x_j \geq i\right) \lor \left(\sum_{j=m+1}^{w} x_j \geq k-i+1\right)\right)
$$

### Phân rã thành các block
Ta sẽ chia dãy $\Omega$ thành các tập con liên tiếp $\omega_1, \omega_2, \ldots, \omega_M$ với độ dài $w$ (đối với tập con cuối cùng có thể có độ dài nhỏ hơn $w$ nếu $n$ không chia hết cho $w$) và có tổng cộng $M = \left\lceil \frac{n}{w} \right\rceil$ tập con. Mỗi tập con sẽ được đánh số từ $1$ đến $M$. Ta sẽ tạo ra các block (khối) từ boundary của các tập con này. Cụ thể, với 2 tập con liên tiếp $\omega_i$ và $\omega_{i+1}$, ta sẽ lấy $x_{i,w}$ và $x_{i+1,1}$ làm boundary và là điểm phân rã (decomposition point) để tạo thành block $B_i$ và $B_{i+1}$. Sau đó ta nối 2 tập con này lại với nhau và duyệt cửa sổ $w$ liên tiếp vào tập mới. Những biến $x$ ở bên trái điểm phân rã sẽ thuộc về block $B_i$ và những biến $x$ ở bên phải điểm phân rã sẽ thuộc về block $B_{i+1}$. Ta sẽ có $\mathcal{P}  = 2*(M-1)$ block sau khi phân rã.


#### Ví dụ
Với $n=10$ và $w=4$, $k=2$, ta sẽ có 3 tập con $$\omega_1 = \{x_1, x_2, x_3, x_4\}$$ $$\omega_2 = \{x_5, x_6, x_7, x_8\}$$ $$\omega_3 = \{x_9, x_{10}\}$$. 

Có 2 boundary là cặp biến $[ x_4 \mid x_5 ]$, $ [x_8 \mid x_9 ]$. Xét 2 tập con $\omega_1$ và $\omega_2$, ta sẽ tạo 2 block $B_1$ và $B_2$ như sau:
$$\omega_1 = [x_1,x_2,x_3,x_4]$$

$$\omega_2 = [x_5,x_6,x_7,x_8]$$

$$ \text{Boundary: } [x_4 \mid x_5]$$

Các constraint crossing boundary:
$$[x_2,x_3,x_4,x_5]$$
$$[x_3,x_4,x_5,x_6]$$
$$[x_4,x_5,x_6,x_7]$$

Sau đó ta sẽ chia các constraint crossing boundary này vào các block tương ứng. Cụ thể, constraint $[x_2,x_3,x_4,x_5]$ sẽ được chia thành 2 phần $[x_2,x_3,x_4]$ thuộc block $B_1$ và $[x_5]$ thuộc block $B_2$. Tương tự, constraint $[x_3,x_4,x_5,x_6]$ sẽ được chia thành 2 phần $[x_3,x_4]$ thuộc block $B_1$ và $[x_5,x_6]$ thuộc block $B_2$. Constraint $[x_4,x_5,x_6,x_7]$ sẽ được chia thành 2 phần $[x_4]$ thuộc block $B_1$ và $[x_5,x_6,x_7]$ thuộc block $B_2$. Như vậy, sau khi phân rã, ta sẽ có 2 block $B_1$ và $B_2$ với các constraint tương ứng như sau:

$$
B_1: [x_1,x_2,x_3,x_4], [x_2,x_3,x_4], [x_3,x_4], [x_4]
$$ 

$$
B_2: [x_5], [x_5,x_6], [x_5,x_6,x_7]
$$

<center>
    <img src="./images/ALSC/fig2.png" alt="Connecting Blocks" width="600"> 
</center>






Với tính chất 1, ta có thể áp dụng SCL để tận dụng các biến phụ. Cách chia cũng tương tự nhưng ta sẽ nâng cấp từ $R_{i,j}$ thành $R_{i,j,s}$ với ý nghĩa là thanh ghi R ở khối $i$ với tổng các biến từ $1$ giá trị thứ $j$ đã đạt được $s$ lần True chưa.
- $R_{i,j,s} = \text{True}$ nếu tổng các biến từ $1$ đến $j$ trong khối $i$ đã đạt được $s$ lần True, nghĩa là $x_1 + x_2 + \ldots + x_j \geq s$.
- $R_{i,j,s} = \text{False}$ nếu tổng các biến từ $1$ đến $j$ trong khối $i$ chưa đạt được $s$ lần True, nghĩa là $x_1 + x_2 + \ldots + x_j \leq s-1$.

***Lưu ý***: 
- Trong một tập con $\omega = {x_{i,1}, x_{i,2}, \ldots, x_{i,j}}$ nếu k < j thì chỉ cần xét đến $R_{i,j,s}$ với $1 \leq s \leq k$
- Biến $m$ trong tính chất 1 chỉ cần là 1 số thoả mãn trong vùng $1 \leq m < w$ để phân rã constraint $x_1+x_2+\ldots+x_w \geq k$ thành các constraint con. Khi đã phân tách thành các khối và kết nối các block lại với nhau, ta chỉ cần các constraint con có thể nối lại với constraint gốc. Xem chi tiết ở mục kết nối các block.

<center>
    <img src="./images/ALSC/fig3.png" alt="ALSC SCL Encoding" width="600">
</center>



### Các ràng buộc của SCL
Với mỗi khối $i$ có độ dài $w_i$, ta có các ràng buộc sau:
\begin{align}
    &\bigwedge_{j=1}^{w_i} \left(x_{i,j} \rightarrow R_{i,j,1}\right) \tag{1} \\
    &\bigwedge_{j=2}^{w_i} \bigwedge_{s=1}^{\min(j-1,k)} \left(R_{i,j-1,s} \rightarrow R_{i,j,s}\right) \tag{2} \\
    &\bigwedge_{j=2}^{w_i} \bigwedge_{s=2}^{\min(j,k)} \left(x_{i,j} \land R_{i,j-1,s-1} \rightarrow R_{i,j,s}\right) \tag{3} \\
    &\bigwedge_{j=1}^{k} \left(\neg x_{i,j} \rightarrow \neg R_{i,j,j}\right) \tag{4} \\
    &\bigwedge_{j=2}^{w_i} \bigwedge_{s=2}^{\min(j,k)} \left(\neg R_{i,j-1,s-1} \rightarrow \neg R_{i,j,s}\right) \tag{5} \\
    &\bigwedge_{j=2}^{w_i} \bigwedge_{s=1}^{\min(j-1,k)} \left(\neg x_{i,j} \land \neg R_{i,j-1,s} \rightarrow \neg R_{i,j,s}\right) \tag{6} \\
    &R_{i,w_i,k} \tag{7}
\end{align}

Trong đó:
- Ràng buộc (1) đảm bảo rằng nếu biến $x_{i,j}$ được gán `True`, thì thanh ghi $R_{i,j,1}$ cũng phải được gán `True`, nghĩa là đã có ít nhất 1 biến True trong các biến từ $1$ đến $j$.
- Ràng buộc (2) đảm bảo rằng nếu đã có ít nhất $s$ biến True trong các biến từ $1$ đến $j-1$, thì cũng phải có ít nhất $s$ biến True trong các biến từ $1$ đến $j$.
- Ràng buộc (3) đảm bảo rằng nếu biến $x_{i,j}$ được gán `True` và đã có ít nhất $s-1$ biến True trong các biến từ $1$ đến $j-1$, thì cũng phải có ít nhất $s$ biến True trong các biến từ $1$ đến $j$.
- Ràng buộc (4) đảm bảo rằng nếu biến $x_{i,j}$ được gán `False`, thì thanh ghi $R_{i,j,j}$ phải được gán `False`, nghĩa là không thể có nhiều hơn $j$ biến True trong các biến từ $1$ đến $j$. 
- Ràng buộc (5) đảm bảo rằng nếu không có ít nhất $s-1$ biến True trong các biến từ $1$ đến $j-1$, thì cũng không thể có ít nhất $s$ biến True trong các biến từ $1$ đến $j$.
- Ràng buộc (6) đảm bảo rằng nếu biến $x_{i,j}$ được gán `False` và không có ít nhất $s$ biến True trong các biến từ $1$ đến $j-1$, thì cũng không thể có ít nhất $s$ biến True trong các biến từ $1$ đến $j$.
- Ràng buộc (7) đảm bảo rằng trong khối $i$ phải có ít nhất $k$ biến True, nghĩa là $R_{i,w_i,k}$ phải được gán `True`. Đây là ràng buộc chính để đảm bảo rằng trong mỗi chuỗi con liên tiếp có độ dài $w$, phải có ít nhất $k$ biến được gán giá trị True. Còn 6 ràng buộc đầu tiên là các ràng buộc phụ để đảm bảo tính nhất quán của các biến phụ $R_{i,j,s}$.

### Kết nối các block
<center>
    <img src="./images/ALSC/fig2.png" alt="Connecting Blocks" width="600">
</center>
Do các block đã được phân rã từ các tập con liên tiếp nên chỉ cần ghép lại các constraint con để ghép thành các constraint gốc trong Sequential Constraints.
Ví dụ: Với constraint $x_2+x_3+x_4+x_5 \geq 2$ crossing boundary giữa block $B_1$ và block $B_2$. Khi đó, constraint gốc có thể biểu diễn thành:

\begin{align*}
    & (x_2+x_3+x_4+x_5 \geq 2) \\
    & \equiv (x_2+x_3+x_4 \geq 1) \lor (x_5 \geq 2) \land \\
    &(x_2+x_3+x_4+x_5 \geq 2) \lor (x_5 \geq 1) \\
    & \equiv (R_{1,3,2}) \land (R_{1,3,1} \lor R_{2,1,1})   
\end{align*}

Tương tự với constraint $x_3+x_4+x_5+x_6 \geq 2$ crossing boundary giữa block $B_1$ và block $B_2$. Khi đó, constraint gốc có thể biểu diễn thành:
\begin{align*}
    & (x_3+x_4+x_5+x_6 \geq 2) \\
    & \equiv \left((x_3+x_4 \geq 1) \lor (x_5+x_6 \geq 2)\right) \land \\
    & \left((x_3+x_4 \geq 2) \lor (x_5+x_6 \geq 1)\right) \\
    & \equiv (R_{1,2,1} \lor R_{2,2,2}) \land (R_{1,2,2} \lor R_{2,2,1})
\end{align*}

#### Công thức nối block trong ALSCE

Các block được tạo thành từng cặp quanh mỗi boundary:
$[(B_1,B_2), (B_3,B_4), \dots]$

Trong đó:
- block lẻ là suffix block,
- block chẵn là prefix block.

Sequential Counter sinh ra register $[R_{i,j,s}]$ với ý nghĩa:
$$
R_{i,j,s}=1
\iff
\text{sub-expression độ dài } j \text{ trong block } i \text{ đạt ít nhất } s $$

Register chỉ tồn tại khi: $1 \le s \le \min(j,k)$

Định nghĩa:

$
\phi(B_i,j,s)=
\begin{cases}
R_{i,j,s}, & 1\le s\le \min(j,k),\\
\textbf{false}, & s>j.
\end{cases}
$

Khi đó công thức nối block tổng quát là:
$$
\bigwedge_{m=1}^{\mathcal{P}}
\;
\bigwedge_{\substack{
a,b\ge1\\
a\le w_{2m-1}\\
b\le w_{2m}\\
a+b=w
}}
\;
\bigwedge_{t=1}^{k}
\left(
\phi(B_{2m-1},a,t)
\vee
\phi(B_{2m},b,k-t+1)
\right)
$$

Ý nghĩa:
- duyệt từng cặp block đối ngẫu,
- duyệt mọi cách split \(a+b=w\),
- nối các register theo tính chất 1.

## Cài đặt với bài toán Nurse Rostering Problem (NRP)

Bài toán **Nurse Rostering Problem (NRP)** yêu cầu xây dựng lịch làm việc khả thi cho $n$ y tá trong $d$ ngày. Với mỗi cặp `(y tá, ngày)`, lịch chọn đúng một trong bốn trạng thái:

| Ký hiệu | Ý nghĩa |
|:--:|---|
| $D$ | Ca ngày |
| $E$ | Ca chiều |
| $N$ | Ca đêm |
| $O$ | Nghỉ |

### Biến quyết định

Với $1 \le i \le n$, $1 \le t \le d$ và $s \in \{D,E,N,O\}$, định nghĩa biến Boolean:

$$
x_{i,t,s} =
\begin{cases}
1, & \text{nếu y tá } i \text{ được gán trạng thái } s \text{ ở ngày } t, \\
0, & \text{ngược lại.}
\end{cases}
$$

Trong đó $x_{i,t,O}=1$ nghĩa là y tá $i$ nghỉ ở ngày $t$. Vì mỗi ngày chọn đúng một trạng thái, điều kiện nghỉ tương đương với việc không làm các ca $D,E,N$.

### Ràng buộc cơ sở
1. Nhiều nhất một ca làm việc mỗi ngày
$$
\begin{aligned}
&\bigwedge_{i=1}^{n} \bigwedge_{t=1}^{d} \left( \sum_{s \in \{D,E,N,O\}} x_{i,t,s} =1 \right) \end{aligned}
$$
2. Nhiều nhất 6 ngày làm việc trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} \neg x_{i,t,O} \leq 6 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} x_{i,t,O} \geq 1 \right) \text{(Dạng At-Least)}
\end{aligned}
$$
Ví dụ biểu diễn

$$
\begin{array}{rcccccccccccccc}
\neg x_{i,1,O} & + & \neg x_{i,2,O} & + & \dots & + & \neg x_{i,7,O} & & & & & & & & \le 6 \\
               &   & \neg x_{i,2,O} & + & \dots & + & \neg x_{i,7,O} & + & \neg x_{i,8,O} & & & & & & \le 6 \\
               &   &                &   & \ddots &   &                &   &                &   & \ddots & & & & \vdots \\
               &   &                &   &        &   & \neg x_{i,d-6,O} & + & \dots          & + & \neg x_{i,d-1,O} & + & \neg x_{i,d,O} & & \le 6
\end{array}
$$

3. Ít nhất 4 ngày nghỉ trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13}  x_{i,t,O} \geq 4 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} \neg x_{i,t,O} \leq 10 \right) \text{(Dạng At-Least)}
\end{aligned}
$$

4. Ít nhất 4 ca chiều trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,E} \geq 4 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,E} \leq 10 \right) \text{(Dạng At-Most)}
\end{aligned}
$$

5. Nhiều nhất 8 ca chiều trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,E} \leq 8 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,E} \geq 6 \right) \text{(Dạng At-Least)}
\end{aligned}
$$

6. Ít nhất 20 ngày làm việc trong mỗi 28 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-27} \left( \sum_{t=j}^{t+27} \neg x_{i,t,O} \geq 20 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-27} \left( \sum_{t=j}^{t+27} x_{i,t,O} \leq 8 \right) \text{(Dạng At-Most)}
\end{aligned} 
$$
Ngoài ra để tổng quát hơn  thì còn có thể thêm ràng buộc dưới đây thành:
$$
x_{i,t,O} \iff \neg (x_{i,t,D} \lor x_{i,t,E} \lor x_{i,t,N}), \quad 1 \leq i \leq n, 1 \leq t \leq d
$$

7. Nhiều nhất 4 ca đêm trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,N} \leq 4 \right) \text{(Dạng At-Most)}\\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,N} \geq 10 \right) \text{(Dạng At-Least) }
\end{aligned} 
$$


8. Ít nhất một ca đêm trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,N} \geq 1 \right) \text{(Dạng At-Least)}\\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,N} \leq 13 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$

9. Ít nhất 2 ca chiều/đêm trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} (x_{i,t,E}+x_{i,t,N}) \geq 2 \right) \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} z_{i,r} \geq 2 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} \neg z_{i,r} \leq 12 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$

Với $1 \leq j \leq d-6$ và $2j-1 \leq r \leq 2(j+6)$:
$$
\begin{cases} 
z_{i,r} = x_{i,j,E} &, r = 2j-1 \\ 
z_{i,r} = x_{i,j,N} &, r = 2j 
\end{cases}
$$ 
$\max(r) = 2d$


10. Nhiều nhất 4 ca chiều/đêm trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} (x_{i,t,E}+x_{i,t,N}) \leq 4 \right) \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} z_{i,r} \leq 4 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} \neg z_{i,r} \geq 10 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$


11. Không được làm ca đêm trong hai ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-1} \left( \sum_{t=j}^{t+1} x_{i,t,N} \leq 1 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-1} \left( \sum_{t=j}^{j+1} \neg x_{i,t,N} \geq 1 \right) \text{(Dạng At-Least) }
\end{aligned} 
$$


### Mã nguồn

In [19]:
from pysat.solvers import Glucose4
import math


class CoreManager:
    def __init__(self, solver=None):
        self.solver = solver if solver is not None else Glucose4()
        self.total_variables = 0
        self.variables = {}
        self.var_names = {}
        self.clauses = []

    def add_variable(self, name):
        self.total_variables += 1
        self.variables[name] = self.total_variables
        self.var_names[self.total_variables] = name
        return self.total_variables

    def get_variable(self, name):
        return self.variables[name]

    def add_clause(self, clause):
        self.clauses.append(clause)
        self.solver.add_clause(clause)

    def solve(self):
        return self.solver.solve()

    def get_model(self):
        return self.solver.get_model()

    def delete(self):
        self.solver.delete()


# Notes: This class uses 1-based indexing for variables and registers to match the formulas in the paper.
# The variable with index 0 is not used and can be considered a dummy variable.
class ALSCE:
    def __init__(self, core_manager):
        self.core_manager = core_manager

    def add_clause(self, clause):
        self.core_manager.add_clause(clause)

    def add_variable(self, name):
        return self.core_manager.add_variable(name)

    def create_subset(self, variables, w):
        subsets = []
        vars = [-1] + variables
        n = len(vars) - 1
        for i in range(1, n + 1, w):
            if i + w > n:
                subset = vars[i:]
            else:
                subset = vars[i:i + w]
            subsets.append(subset)
        return subsets

    def create_block(self, block_index, subset, K, enforced_at_last):
        if len(subset) == 0:
            raise ValueError("Block must contain at least one variable")
        if K < 1:
            raise ValueError("K must be positive")

        registers = [[0 for _ in range(K + 1)]]
        x = [0] + subset
        w_i = len(x) - 1

        for j in range(1, w_i + 1):
            register = [0]
            for s in range(1, min(j, K) + 1):
                register_name = f"R_{block_index}_{j}_{s}"
                register.append(self.add_variable(register_name))
            registers.append(register)

        # (1) x_j -> R_j,1
        for j in range(1, w_i + 1):
            self.add_clause([-x[j], registers[j][1]])

        # (2) R_{j-1,s} -> R_{j,s}
        for j in range(2, w_i + 1):
            for s in range(1, min(j - 1, K) + 1):
                self.add_clause([-registers[j - 1][s], registers[j][s]])

        # (3) x_j and R_{j-1,s-1} -> R_{j,s}
        for j in range(2, w_i + 1):
            for s in range(2, min(j, K) + 1):
                self.add_clause([-x[j], -registers[j - 1][s - 1], registers[j][s]])

        # (4) not x_j -> not R_{j,j}
        for j in range(1, min(w_i, K) + 1):
            self.add_clause([x[j], -registers[j][j]])

        # (5) not R_{j-1,s-1} -> not R_{j,s}
        for j in range(2, w_i + 1):
            for s in range(2, min(j, K) + 1):
                self.add_clause([registers[j - 1][s - 1], -registers[j][s]])

        # (6) not x_j and not R_{j-1,s} -> not R_{j,s}
        for j in range(2, w_i + 1):
            for s in range(1, min(j - 1, K) + 1):
                self.add_clause([x[j], registers[j - 1][s], -registers[j][s]])

        # (7) Whole block must contain at least K true literals when it represents a complete window.
        if enforced_at_last:
            if K > w_i:
                self.add_clause([])
            else:
                self.add_clause([registers[w_i][K]])

        return registers

    def create_blocks(self, subsets, w, K):
        if len(subsets) == 1:
            block = self.create_block(1, list(reversed(subsets[0])), K, True)
            return [None, block]

        blocks = [None]
        for i in range(0, len(subsets) - 1):
            RL_subset = list(reversed(subsets[i]))
            LR_subset = subsets[i + 1]

            RL_block = self.create_block(len(blocks), RL_subset, K, True)
            blocks.append(RL_block)

            LR_block = self.create_block(len(blocks), LR_subset, K, len(LR_subset) == w)
            blocks.append(LR_block)

        return blocks

    def connect_blocks(self, blocks, w, K):
        def phi(block, j, s):
            n = len(block) - 1
            if 1 <= j <= n and 1 <= s <= min(j, K) and block[j][s] != 0:
                return block[j][s]
            return None

        len_blocks = len(blocks) - 1
        for i in range(1, len_blocks, 2):
            RL_block = blocks[i]
            LR_block = blocks[i + 1]
            len_RL_block = len(RL_block) - 1
            len_LR_block = len(LR_block) - 1

            for a in range(1, len_RL_block + 1):
                b = w - a
                if b < 1 or b > len_LR_block:
                    continue

                for t in range(1, K + 1):
                    clause = []
                    left_lit = phi(RL_block, a, t)
                    right_lit = phi(LR_block, b, K - t + 1)

                    if left_lit is not None:
                        clause.append(left_lit)
                    if right_lit is not None:
                        clause.append(right_lit)

                    self.add_clause(clause)

    def alsc(self, variables, w, k):
        if not (1 < w <= len(variables)):
            raise ValueError("ALSC requires 1 < w <= len(variables)")
        if not (1 <= k <= w):
            raise ValueError("ALSC requires 1 <= k <= w")

        subsets = self.create_subset(variables, w)
        blocks = self.create_blocks(subsets, w, k)
        self.connect_blocks(blocks, w, k)



#### Sequential Encoding for At-Least K Constraints

In [20]:
class SCE:
    def __init__(self, core_manager:CoreManager):
        self.core_manager = core_manager
    
    def add_clause(self, clause:list):
        self.core_manager.add_clause(clause)
    
    def new_counter_states(self, variables:list, K:int)->dict:
        registers = {}

        for i in range(len(variables)-1):
            for j in range(min(i+1,K)):
                var_name = f"sce_r_{i}_{j+1}"
                registers[(i,j)] = self.core_manager.add_variable(var_name)

        return registers

    def amk(self, variables:list, K:int)->dict:
        n = len(variables)

        if K < 0:
            self.add_clause([])
            return {}

        if K >= n:
            return {}

        if K == 0:
            for x in variables:
                self.add_clause([-x])
            return {}

        registers = self.new_counter_states(variables,K)

        # Formula 1
        for i in range(n-1):
            self.add_clause([-variables[i],registers[(i,0)]])

        # Formula 2
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([-registers[(i-1,j)],registers[(i,j)]])

        # Formula 3
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([-variables[i],-registers[(i-1,j-1)],registers[(i,j)]])

        # Formula 8
        for i in range(K,n):
            self.add_clause([-variables[i],-registers[(i-1,K-1)]])

        return registers

    def alk(self, variables:list, K:int)->dict:
        n = len(variables)

        if K <= 0:
            return {}

        if K > n:
            self.add_clause([])
            return {}

        if K == n:
            for x in variables:
                self.add_clause([x])
            return {}

        registers = self.new_counter_states(variables,K)

        # Formula 1
        for i in range(n-1):
            self.add_clause([-variables[i],registers[(i,0)]])

        # Formula 2
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([-registers[(i-1,j)],registers[(i,j)]])

        # Formula 3
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([-variables[i],-registers[(i-1,j-1)],registers[(i,j)]])

        # Formula 4
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([variables[i],registers[(i-1,j)],-registers[(i,j)]])

        # Formula 5
        for i in range(min(K,n-1)):
            self.add_clause([variables[i],-registers[(i,i)]])

        # Formula 6
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([registers[(i-1,j-1)],-registers[(i,j)]])

        # Formula 7
        last_x = variables[-1]
        last_prefix = n-2
        if K == 1:
            self.add_clause([registers[(last_prefix,0)],last_x])
        else:
            self.add_clause([registers[(last_prefix,K-1)],last_x])
            self.add_clause([registers[(last_prefix,K-1)],registers[(last_prefix,K-2)]])

        return registers

    def exk(self, variables:list, K:int)->dict:
        return {
            "at_least": self.alk(variables,K),
            "at_most": self.amk(variables,K),
        }

    def range(self, variables:list, u:int, v:int)->dict:
        return {
            "lower": self.alk(variables,u),
            "upper": self.amk(variables,v),
        }



In [ ]:
class NurseRosteringProblem:
    """NRP model whose constraints follow the formulas in Cell 3."""

    SHIFTS = ("D", "E", "N", "O")

    def __init__(self, nurse_count: int, day_count: int):
        if nurse_count <= 0:
            raise ValueError("nurse_count must be positive")
        if day_count <= 0:
            raise ValueError("day_count must be positive")

        self.nurse_count = nurse_count
        self.day_count = day_count
        self.core = CoreManager()
        self.sce = SCE(self.core)
        self.alsce = ALSCE(self.core)
        self.x = {}
        self._solved = False
        self._is_sat = None

        self._create_decision_variables()

    def _create_decision_variables(self):
        for nurse in range(1, self.nurse_count + 1):
            for day in range(1, self.day_count + 1):
                for shift in self.SHIFTS:
                    name = f"x_{nurse}_{day}_{shift}"
                    self.x[(nurse, day, shift)] = self.core.add_variable(name)

    def shift_var(self, nurse: int, day: int, shift: str) -> int:
        return self.x[(nurse, day, shift)]

    def day_vars(self, nurse: int, day: int) -> list[int]:
        return [self.shift_var(nurse, day, shift) for shift in self.SHIFTS]

    def shift_sequence(self, nurse: int, shift: str) -> list[int]:
        return [
            self.shift_var(nurse, day, shift)
            for day in range(1, self.day_count + 1)
        ]

    def evening_night_window(self, nurse: int, start_day: int, width: int = 7) -> list[int]:
        literals = []
        for day in range(start_day, start_day + width):
            literals.append(self.shift_var(nurse, day, "E"))
            literals.append(self.shift_var(nurse, day, "N"))
        return literals

    def shift_coverage(self, day: int, shift: str) -> list[int]:
        return [
            self.shift_var(nurse, day, shift)
            for nurse in range(1, self.nurse_count + 1)
        ]

    def add_sliding_sce_at_least(self, literals: list[int], width: int, lower: int):
        if len(literals) < width:
            return
        for start in range(len(literals) - width + 1):
            self.sce.alk(literals[start:start + width], lower)

    def add_sliding_sce_at_most(self, literals: list[int], width: int, upper: int):
        if len(literals) < width:
            return
        for start in range(len(literals) - width + 1):
            self.sce.amk(literals[start:start + width], upper)

    def add_alsc_at_least(self, literals: list[int], width: int, lower: int):
        if len(literals) >= width:
            self.alsce.alsc(literals, width, lower)

    def add_alsc_at_most(self, literals: list[int], width: int, upper: int):
        if len(literals) >= width:
            self.alsce.alsc([-literal for literal in literals], width, width - upper)

    def add_daily_coverage(self, min_demand=None, max_demand=None):
        min_demand = min_demand or {}
        max_demand = max_demand or {}

        for day in range(1, self.day_count + 1):
            for shift, lower in min_demand.items():
                self.sce.alk(self.shift_coverage(day, shift), lower)
            for shift, upper in max_demand.items():
                self.sce.amk(self.shift_coverage(day, shift), upper)

    def add_constraints_for_nurse(self, nurse: int):
        off = self.shift_sequence(nurse, "O")
        evening = self.shift_sequence(nurse, "E")
        night = self.shift_sequence(nurse, "N")

        # (1) sum_{s in {D,E,N,O}} x_{i,t,s} = 1.
        for day in range(1, self.day_count + 1):
            self.sce.exk(self.day_vars(nurse, day), 1)

        # (2) sum_{t=j}^{j+6} not O <= 6 <=> sum O >= 1.
        self.add_alsc_at_least(off, 7, 1)

        # (3) sum_{t=j}^{j+13} O >= 4.
        self.add_alsc_at_least(off, 14, 4)

        # (4) sum_{t=j}^{j+13} E >= 4.
        self.add_alsc_at_least(evening, 14, 4)

        # (5) sum_{t=j}^{j+13} E <= 8.
        self.add_alsc_at_most(evening, 14, 8)

        # (6) sum working >= 20 <=> sum O <= 8 over every 28 days.
        self.add_alsc_at_most(off, 28, 8)

        # (7) sum_{t=j}^{j+13} N <= 4.
        self.add_alsc_at_most(night, 14, 4)

        # (8) sum_{t=j}^{j+13} N >= 1.
        self.add_alsc_at_least(night, 14, 1)

        # (9), (10) 2 <= sum_{t=j}^{j+6}(E + N) <= 4.
        en_literals = self.evening_night_window(nurse, 1, width=self.day_count)
        # Each 7-day window contains 14 literals: E_j, N_j, ..., E_{j+6}, N_{j+6}.
        self.add_alsc_at_least(en_literals, 14, 2)
        self.add_alsc_at_most(en_literals, 14, 4)

        # (11) sum_{t=j}^{j+1} N <= 1.
        self.add_alsc_at_most(night, 2, 1)

    def add_constraints(self, min_demand=None, max_demand=None):
        for nurse in range(1, self.nurse_count + 1):
            self.add_constraints_for_nurse(nurse)

        # Optional coverage constraints are not part of Cell 3, but can be added for demand.
        self.add_daily_coverage(min_demand=min_demand, max_demand=max_demand)
        return self

    def solve(self) -> bool:
        if not self._solved:
            self._is_sat = self.core.solve()
            self._solved = True
        return self._is_sat

    def model(self) -> list[int] | None:
        if not self.solve():
            return None
        return self.core.get_model()

    def schedule(self) -> list[list[str]] | None:
        model = self.model()
        if model is None:
            return None

        positive = {literal for literal in model if literal > 0}
        schedule = []
        for nurse in range(1, self.nurse_count + 1):
            row = []
            for day in range(1, self.day_count + 1):
                row.append(next(
                    shift for shift in self.SHIFTS
                    if self.shift_var(nurse, day, shift) in positive
                ))
            schedule.append(row)
        return schedule


def solve_nrp(nurse_count=10, day_count=28, min_demand=None, max_demand=None):
    nrp = NurseRosteringProblem(nurse_count, day_count).add_constraints(
        min_demand=min_demand,
        max_demand=max_demand,
    )
    return nrp, nrp.schedule()


def count_windows(row: list[str], width: int, predicate) -> list[int]:
    return [
        sum(1 for shift in row[start:start + width] if predicate(shift))
        for start in range(len(row) - width + 1)
    ]


def validate_nrp_schedule(schedule, day_count, min_demand=None, max_demand=None):
    checks = []
    min_demand = min_demand or {}
    max_demand = max_demand or {}

    for nurse_id, row in enumerate(schedule, start=1):
        checks.append((
            f"Nurse {nurse_id}: (1) exactly one status per day",
            len(row) == day_count and all(shift in NurseRosteringProblem.SHIFTS for shift in row),
        ))

        if day_count >= 7:
            checks.append((f"Nurse {nurse_id}: (2) >= 1 O / 7", all(v >= 1 for v in count_windows(row, 7, lambda s: s == "O"))))
            checks.append((f"Nurse {nurse_id}: (9) >= 2 E/N / 7", all(v >= 2 for v in count_windows(row, 7, lambda s: s in {"E", "N"}))))
            checks.append((f"Nurse {nurse_id}: (10) <= 4 E/N / 7", all(v <= 4 for v in count_windows(row, 7, lambda s: s in {"E", "N"}))))

        if day_count >= 14:
            checks.append((f"Nurse {nurse_id}: (3) >= 4 O / 14", all(v >= 4 for v in count_windows(row, 14, lambda s: s == "O"))))
            checks.append((f"Nurse {nurse_id}: (4) >= 4 E / 14", all(v >= 4 for v in count_windows(row, 14, lambda s: s == "E"))))
            checks.append((f"Nurse {nurse_id}: (5) <= 8 E / 14", all(v <= 8 for v in count_windows(row, 14, lambda s: s == "E"))))
            checks.append((f"Nurse {nurse_id}: (7) <= 4 N / 14", all(v <= 4 for v in count_windows(row, 14, lambda s: s == "N"))))
            checks.append((f"Nurse {nurse_id}: (8) >= 1 N / 14", all(v >= 1 for v in count_windows(row, 14, lambda s: s == "N"))))

        if day_count >= 28:
            checks.append((f"Nurse {nurse_id}: (6) >= 20 working / 28", all(v >= 20 for v in count_windows(row, 28, lambda s: s != "O"))))

        checks.append((
            f"Nurse {nurse_id}: (11) no consecutive N",
            all(not (row[d] == "N" and row[d + 1] == "N") for d in range(len(row) - 1)),
        ))

    for day in range(day_count):
        for shift, lower in min_demand.items():
            count = sum(row[day] == shift for row in schedule)
            checks.append((f"Day {day + 1}: >= {lower} {shift}", count >= lower))
        for shift, upper in max_demand.items():
            count = sum(row[day] == shift for row in schedule)
            checks.append((f"Day {day + 1}: <= {upper} {shift}", count <= upper))

    return checks


nrp, schedule = solve_nrp(nurse_count=30, day_count=84)

print("SAT:", schedule is not None)
print("Variables:", nrp.core.total_variables)
print("Clauses:", len(nrp.core.clauses))

if schedule is not None:
    for nurse_id, row in enumerate(schedule, start=1):
        print(f"Nurse {nurse_id}:", " ".join(row))

    validation = validate_nrp_schedule(schedule, nrp.day_count)
    failed = [name for name, ok in validation if not ok]
    print("Validation checks:", len(validation))
    print("Failed:", len(failed))
    print("All constraints satisfied:", len(failed) == 0)
    for name in failed:
        print("FAIL -", name)


SAT: True
Variables: 243060
Clauses: 982860
Nurse 1: D O N D E O E N O D E D E O E O N D E O E D O E E D N O D O E E D O N E O E D D D O E O E E D O N D O E E E D O E O E N D O E N O D D E D O E O E N E O D D O E E E E O
Nurse 2: N O O O D E D E E O N D E D E O O O D E D E N O D D E E D O O O N E E D D O E D D E N O O O E D E D N O E D E E D O O O E N D E D O N E D E E O O O E E N D D O D E D N
Nurse 3: E E N D N O O E N D O E O D E E E D N O O E E E O D O E D D E E N O O D E D O E O D N D E E E O O D N E O E O D D N D E E O O E N D O E O E D E D D E O O N E N O D O E
Nurse 4: O D E E N D D O N D O E E O O E D E N E D O D E O E N O O D E E N D D O E E O E E O O D D N E N E O D D O E E O O E D D E N D O E N O E N O O E D E D D D O E E O E N O
Nurse 5: O O N D O E D N E D E O E D O O N E O N D D E D E O E D O O E E O N E D D E E O D E O O E N O D E D D E E O E E O O D N O E D D N D E O E E O O N E O E N D D N E O E D
Nurse 6: E D O E N E O E O D O E D D E N O E D E O N O D O E E D D N O 